# Random Forest - viele unterschiedliche Bäume

Bootstrap-Stichproben und zufällige Merkmalsmengen erzeugen verschiedene Bäume. Mehrheitsentscheidung reduziert die Varianz des Einzelbaums.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score,roc_auc_score
from sklearn.model_selection import GridSearchCV,cross_validate,train_test_split
from sklearn.tree import DecisionTreeClassifier
X,y=make_classification(n_samples=1100,n_features=10,n_informative=5,n_redundant=2,weights=[.65,.35],class_sep=.9,flip_y=.04,random_state=42)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)

rng=np.random.default_rng(42); gezogen=rng.choice(len(X_train),len(X_train),replace=True)
print("Verschiedene Bootstrap-Zeilen:",len(np.unique(gezogen)),"von",len(X_train),"; OOB-Anteil:",round(1-len(np.unique(gezogen))/len(X_train),3))

## Hyperparameter

| Parameter | Bedeutung |
|---|---|
| `n_estimators` | Zahl der Bäume; mehr stabilisiert, kostet Zeit |
| `max_features` | geprüfte Merkmale je Split; steuert Vielfalt |
| `max_depth`, `min_samples_leaf` | Komplexität der Einzelbäume |
| `bootstrap` | Ziehen mit Zurücklegen |
| `oob_score` | interne Bewertung auf nicht gezogenen Zeilen |
| `class_weight` | Gewichte bei ungleichen Klassen |
| `n_jobs` | Parallelisierung; `-1` nutzt alle Kerne |

In [ ]:
def metriken(name, modell, X_test, y_test):
    pred = modell.predict(X_test)
    prob = modell.predict_proba(X_test)[:, 1]
    return {"Modell": name, "Accuracy": accuracy_score(y_test,pred),
            "Precision": precision_score(y_test,pred), "Recall": recall_score(y_test,pred),
            "F1": f1_score(y_test,pred), "ROC-AUC": roc_auc_score(y_test,prob)}

baum=DecisionTreeClassifier(random_state=42).fit(X_train,y_train)
forest=RandomForestClassifier(n_estimators=250,oob_score=True,random_state=42,n_jobs=-1).fit(X_train,y_train)
display(pd.DataFrame([metriken("Einzelbaum",baum,X_test,y_test),metriken("Forest",forest,X_test,y_test)]).set_index("Modell").round(3)); print("OOB-Accuracy:",round(forest.oob_score_,3))

ns=[20,50,100,200,350]; oob=[]
for n in ns:
    m=RandomForestClassifier(n_estimators=n,oob_score=True,random_state=42,n_jobs=-1).fit(X_train,y_train); oob.append(m.oob_score_)
plt.plot(ns,oob,marker="o"); plt.xlabel("n_estimators"); plt.ylabel("OOB-Accuracy"); plt.grid(alpha=.3); plt.show()

## Kleine Grid Search und Stabilitätsvergleich

In [ ]:
grid={"n_estimators":[100,250],"max_depth":[None,6,12],"min_samples_leaf":[1,4],"max_features":["sqrt",.7]}
suche=GridSearchCV(RandomForestClassifier(random_state=42,n_jobs=-1),grid,scoring="f1",cv=4,n_jobs=-1).fit(X_train,y_train)
print("Beste Parameter:",suche.best_params_)
display(pd.DataFrame([metriken("Einzelbaum",baum,X_test,y_test),metriken("Standard-Forest",forest,X_test,y_test),metriken("Grid Search",suche.best_estimator_,X_test,y_test)]).set_index("Modell").round(3))
stab=[]
for n,m in {"Einzelbaum":baum,"Forest":suche.best_estimator_}.items():
    s=cross_validate(m,X_train,y_train,cv=5,scoring="f1")["test_score"]; stab.append({"Modell":n,"CV-F1 Mittel":s.mean(),"CV-F1 Std":s.std()})
display(pd.DataFrame(stab).set_index("Modell").round(3))
pd.Series(suche.best_estimator_.feature_importances_,index=[f"Merkmal {i}" for i in range(10)]).sort_values().plot.barh(color="forestgreen"); plt.show()

Impurity Importance ist keine Kausalität und kann Merkmale mit vielen Splitpunkten bevorzugen. Permutation Importance auf Validierungsdaten ist eine sinnvolle Ergänzung.